#### 학습목표
- 당연히 LLM, LangChain 복습
- 로그 데이터 분석 및 시각화
- 정규표현식
- 분석결과 불필요한 정보 전처리

In [4]:
# llm
import os
import openai
from   openai import OpenAI
from   dotenv import load_dotenv

# langchain 
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.chat_models       import ChatOpenAI 
from langchain.vectorstores      import FAISS
from langchain.text_splitter     import CharacterTextSplitter
from langchain.chains            import RetrievalQA

- 정규표현식 

In [ ]:
'''
모듈 re
정규 표현식이란?
- 문자열 패턴을 만들어서 문자열로부터 검색, 추출, 변환을 도와주는 형식 

패턴을 만드는 방법 
- 메타문자(., ^, $, *, +, ?, a{3}, [abc], (abc)+ )
- 패턴(\d, \D, \w, \W, \s, \S) 

re.search() : 부분일치(텍스트 내에서 첫 매치)
re.match()  : 문자열 시작에서 매치
re.fullmatch() : 전체 문자열이 패턴과 일치할 때
re.sub() : 치환 
re.findall() : 문자열 전체에서 모든 일치하는 부분을 리스트로 반환  

re.sub(r"^```[a-zA-Z]*\n?", "", csv_text)  # ```csv 또는 ``` 제거
re.sub(r"```$", "", csv_text).strip()       # 닫는 ``` 제거

'''

In [8]:
import re

txt = '문의 이메일은 jslim9413@naver.com 입니다.' 

pattern = r'\w+@\w+\.\w+' 

result = re.search(pattern, txt) 
if result :
    print(result.group() )

jslim9413@naver.com


In [15]:
import re

txt = '문의하신 고객의 전화번호는 010-1234-5678 이고,  이메일은 jslim9413@naver.com 입니다.' 

# 전화번호 추출 패턴
patternPhone = r'01[0-9]-\d{4}-\d{4}'
phone = re.findall(patternPhone, txt) 
print('type - ' , type(phone))
print(phone[0])

# 이메일 추출 패턴
patternEmail = r'\w+@\w+\.\w+'
email = re.search(patternEmail, txt) 
if result :
    print(email.group() )

# Q
# 숫자만 추출한다면?
lst = re.findall(r'\d+' , txt)
print(lst)

type -  <class 'list'>
010-1234-5678
jslim9413@naver.com
['010', '1234', '5678', '9413']


In [ ]:
'''
AI Model 반환결과는 자유로운 영혼의 문자열
필요에 따라서 특정 조건에 만족하는 데이터 추출, 변환 etc....
이럴경우 re 이용해서 처리
'''

In [63]:
# llm
import os, re, openai, json 
from   openai import OpenAI
from   dotenv import load_dotenv

# langchain 
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.chat_models       import ChatOpenAI 
from langchain.prompts           import ChatPromptTemplate
from langchain.vectorstores      import FAISS
from langchain.text_splitter     import CharacterTextSplitter
from langchain.chains            import RetrievalQA, LLMChain


In [17]:
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY') 

def masking(key) :
    # 앞 4자리, 뒤 4자리만 남기고 * 처리 
    if len(key) <= 8 :
        return '*' * len(key)
    return  key[ : 4 ] + "*" * (len(key) - 8 ) + key[ -4 : ] 
    
masked_api_key = masking(api_key)
print('masked api key : ' , masked_api_key) 

masked api key :  sk-p************************************************************************************************************************************************************dv0A


In [18]:
'''
temperature(0 ~ 2) 
- 값이 낮을수록 응답은 보수적(같은 입력이면 거의 같은 답변)
- 값이 높을수록 응답은 창의적(같은 입력에서도 다양한 답변)
'''
chat_uncreative = ChatOpenAI(model_name="gpt-3.5-turbo" , temperature=0) 

chat_creative   = ChatOpenAI(model_name="gpt-3.5-turbo" , temperature=0.8)


C:\Users\user\AppData\Local\Temp\ipykernel_6432\353013674.py:6: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  chat_uncreative = ChatOpenAI(model_name="gpt-3.5-turbo" , temperature=0)


In [19]:
prompt = "인공지능을 간단히 설명해줘." 
print('uncreative - ')
print('chat_uncreative predict - ' , chat_uncreative.predict(prompt))
print()
print('creative - ')
print('chat_creative predict   - ' , chat_creative.predict(prompt))


uncreative - 


C:\Users\user\AppData\Local\Temp\ipykernel_6432\2769937924.py:3: LangChainDeprecationWarning: The method `BaseChatModel.predict` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use invoke instead.
  print('chat_uncreative predict - ' , chat_uncreative.predict(prompt))


chat_uncreative predict -  인공지능은 인간의 학습, 추론, 판단 등의 능력을 컴퓨터 프로그램이나 기계에 구현한 기술을 말합니다. 이를 통해 기계가 인간과 유사한 지능적인 작업을 수행할 수 있게 됩니다. 인공지능은 머신러닝, 딥러닝, 자연어 처리 등 다양한 기술을 활용하여 다양한 분야에서 활용되고 있습니다.

creative - 
chat_creative predict   -  인공지능은 컴퓨터 시스템이 인간의 학습, 추론, 문제 해결 및 결정을 수행하는 능력을 가진 기술을 말합니다. 이를 위해 기계 학습, 딥러닝, 자연어 처리 등의 다양한 기술이 활용됩니다. 인공지능은 사람의 지능적인 작업을 자동화하고, 새로운 문제에 대한 해결책을 찾아내는 등 다양한 분야에서 활용되고 있습니다.


In [20]:
prompt = "인공지능을 간단히 설명해줘." 
print('uncreative - ')
print('chat_uncreative predict - ' , chat_uncreative.predict(prompt))
print()
print('creative - ')
print('chat_creative predict   - ' , chat_creative.predict(prompt))

uncreative - 
chat_uncreative predict -  인공지능은 인간의 학습, 추론, 판단 등의 능력을 컴퓨터 프로그램이나 기계에 구현한 기술을 말합니다. 이를 통해 기계가 인간과 유사한 지능적인 작업을 수행할 수 있게 됩니다. 인공지능은 머신러닝, 딥러닝, 자연어 처리 등 다양한 기술을 활용하여 다양한 분야에서 활용되고 있습니다.

creative - 
chat_creative predict   -  인공지능은 컴퓨터 프로그램이나 기계가 인간의 학습, 추론, 문제 해결 등의 지능적인 작업을 수행할 수 있는 기술을 말합니다. 이를 통해 인공지능은 데이터를 분석하고 학습하여 패턴을 파악하거나 문제를 해결할 수 있게 됩니다. 인공지능은 다양한 분야에서 활용되고 있으며, 더 나은 의사결정을 내리고 생산성을 높이는 등의 기능을 제공합니다.


In [25]:
# langchain llm model 
chat = ChatOpenAI(model_name="gpt-3.5-turbo" , temperature=0)
# langchain prompt template
question = '섭섭님과 상담이 필요하면 jslim9413@naver.com 또는 jslim9413@gmail.com 으로 문의 부탁드립니다.' 
prompt = ChatPromptTemplate.from_template(
    "해당 텍스트에서 미사어구를 넣어서 좋은 문장을 만들어줘 : {text} "
)
# langchain except rag
chain = LLMChain(llm=chat, prompt=prompt)

In [30]:
# chain 생성 후 실행(run) 
answer = chain.run(text=question)
print(answer)
print()
print('re - ')
emails = re.findall(r'\w+@\w+\.\w+' , answer) 
print(emails)

"오늘 날씨가 정말 좋아서 나들이를 가기에 딱 좋은 날이야."

re - 
[]


In [ ]:
'''
Quiz) 아래 제공되는 입력정보를 바탕으로 langchain 모델을 만들고 실행해 본다.
- 조건) 프롬프트는 : json 형식으로 정리해줘
- 조건) llm의 반환값을 확인하고 json 형식인지 확인하고 해당 결과를 정규표현식을 사용하여 데이터만 추출하고
- 조건) 구조화된 json 데이터를 만들기(json.dumps()) 
'''

reservation = """
이름  : 임섭순
연락처: 010-1234-5678
예약일: 2025-11-15
문의 이메일: jslim9413@naver.com
추가 이메일: jslim9413@gmail.com
"""

In [50]:
# langchain llm model 
chat   = ChatOpenAI(model_name="gpt-4o-mini" , temperature=0)
prompt = ChatPromptTemplate.from_template(
    "입력받은 텍스를 이름,이메일, 전화번호, 날짜를 추출하고 json 형식으로 정리해줘:{text}"
)
chain = LLMChain(llm=chat, prompt=prompt)

In [66]:
reservation = """
이름  : 임섭순
연락처: 010-1234-5678
예약일: 2025-11-15
문의 이메일: jslim9413@naver.com
추가 이메일: jslim9413@gmail.com
"""
answer = chain.run(text=reservation)
print(answer)

pattern = r'json\s*(\{[\s\S]*?\})'

match = re.search(pattern, answer)
if match : 
    result = match.group(1)
    print(">>>>>>>>>>>>>>>>>>>>>>>> result")
    print(result)
    # json.dumps() (dict -> json ), json.loads( json(text) -> dict ) 
    print('type - ' , type(json.loads(result)))
    print(json.loads(result))

입력받은 텍스트에서 이름, 이메일, 전화번호, 날짜를 추출하여 JSON 형식으로 정리한 결과는 다음과 같습니다:

```json
{
  "이름": "임섭순",
  "연락처": "010-1234-5678",
  "예약일": "2025-11-15",
  "문의 이메일": "jslim9413@naver.com",
  "추가 이메일": "jslim9413@gmail.com"
}
```
>>>>>>>>>>>>>>>>>>>>>>>> result
{
  "이름": "임섭순",
  "연락처": "010-1234-5678",
  "예약일": "2025-11-15",
  "문의 이메일": "jslim9413@naver.com",
  "추가 이메일": "jslim9413@gmail.com"
}
type -  <class 'dict'>
{'이름': '임섭순', '연락처': '010-1234-5678', '예약일': '2025-11-15', '문의 이메일': 'jslim9413@naver.com', '추가 이메일': 'jslim9413@gmail.com'}
